# Geomar AI Oxygen - Hypoxia Prediction Training (Google Colab)

This notebook runs the weighted hypoxia prediction model training pipeline on Google Colab.

**Project**: Boknis Eck hypoxia prediction using weighted Temporal Fusion Transformer  
**Repository**: https://github.com/YOUR_USERNAME/Geomar_AI_Oxygen

## Notebook Overview

1. **Setup**: Install dependencies, clone repository, mount Google Drive
2. **Data Preparation**: Verify data pipeline
3. **Training Options**: Quick test / Full training / Hyperparameter tuning
4. **Results**: View metrics, download checkpoints

## Requirements

- GPU runtime REQUIRED (Runtime > Change runtime type > GPU)
- Google Drive for checkpoints (optional but recommended)
- **Standard RAM** is sufficient (DO NOT select High-RAM)

# 1. Setup Environment

In [ ]:
# Check GPU availability and Colab pre-installed versions
import torch
import pandas as pd
import numpy as np

print("="*80)
print("SYSTEM INFO")
print("="*80)
print(f"\nPyTorch: {torch.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem_gb:.1f} GB")
    
    # Adjust batch size based on GPU
    if "A100" in gpu_name:
        RECOMMENDED_BATCH_SIZE = 128
        print(f"\n✓ A100 detected! Recommended batch size: {RECOMMENDED_BATCH_SIZE}")
    else:
        RECOMMENDED_BATCH_SIZE = 64
        print(f"\n✓ GPU detected! Recommended batch size: {RECOMMENDED_BATCH_SIZE}")
else:
    print("\n⚠️  NO GPU DETECTED - Training will be VERY slow!")
    print("   Go to: Runtime > Change runtime type > Hardware accelerator > GPU")
    RECOMMENDED_BATCH_SIZE = 32

In [ ]:
# Clone repository
import os

REPO_URL = "https://github.com/YOUR_USERNAME/Geomar_AI_Oxygen.git"  # ⚠️ UPDATE THIS
REPO_NAME = "Geomar_AI_Oxygen"

if os.path.exists(REPO_NAME):
    print(f"Repository exists. Pulling latest changes...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning from {REPO_URL}...")
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"\n✓ Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies (Colab-compatible versions)
print("Installing dependencies...\n")
print("Note: Some version warnings are expected and safe to ignore.\n")

# Use Colab-compatible requirements
!pip install -q pytorch-forecasting==1.7.0
!pip install -q 'lightning>=2.0.0,<2.7.0'
!pip install -q 'wetterdienst>=0.90.0'
!pip install -q 'plotly>=5.0.0'
!pip install -q 'optuna>=3.0.0'

print("\n" + "="*80)
print("VERIFYING INSTALLATION")
print("="*80 + "\n")

try:
    import pytorch_forecasting
    import lightning.pytorch as pl
    import optuna
    from wetterdienst import Wetterdienst
    
    print("✓ All key packages imported successfully!\n")
    print(f"Package versions:")
    print(f"  pytorch-forecasting: {pytorch_forecasting.__version__}")
    print(f"  lightning: {pl.__version__}")
    print(f"  optuna: {optuna.__version__}")
    print(f"  torch: {torch.__version__}")
    print(f"  pandas: {pd.__version__}")
    print(f"  numpy: {np.__version__}")
    
except ImportError as e:
    print(f"\n❌ Import error: {e}")
    print("\nTry restarting runtime: Runtime > Restart runtime")

In [ ]:
# Mount Google Drive for checkpoint persistence (RECOMMENDED)
from google.colab import drive

MOUNT_DRIVE = True  # Set to False to skip (checkpoints will be lost on session end)

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/Geomar_Checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"\n✓ Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")
    print("  (These will persist even after session ends)")
else:
    CHECKPOINT_DIR = 'models/hypoxia_tft'
    print(f"\n⚠️  Checkpoints will be saved locally: {CHECKPOINT_DIR}")
    print("  (These will be LOST when session ends! Download before closing.)")

# 2. Verify Data and Pipeline

In [ ]:
# Verify data files and pipeline
print("Checking data files...\n")
!ls -lh Documentation/data/

print("\nTesting data ingestion...")
from src import data_ingestion

df_ocean = data_ingestion._load_ocean_data()
print(f"\n✓ Ocean data loaded: {len(df_ocean)} rows")
print(f"  Date range: {df_ocean['Date'].min()} to {df_ocean['Date'].max()}")
print(f"  Depths: {sorted(df_ocean['Depth_m'].unique())}")

In [ ]:
# Run unit tests
print("Running unit tests...\n")
!pip install -q pytest
!python -m pytest tests/ -v --tb=short

print("\n✓ All tests passed! Pipeline is ready for training.")

# 3. Training Options\n\n**Choose ONE of the following:**\n- Option A: Quick Test (5 epochs, ~2-5 min)\n- Option B: Full Training (100 epochs with early stopping, ~10-60 min depending on GPU)\n- Option C: Hyperparameter Tuning (LONG: 1-20 hours depending on GPU and trials)\n- Option D: Weighted Loss Verification (~5-10 min)

## Option A: Quick Test Training (5 epochs)

In [ ]:
# Quick test - verify everything works
print("Starting quick test (5 epochs)...\n")

!python train.py \
    --max-epochs 5 \
    --batch-size 32 \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --patience 2

print("\n✓ Quick test complete!")

## Option B: Full Training

In [ ]:
# Full training with default hyperparameters
print(f"Starting full training (batch_size={RECOMMENDED_BATCH_SIZE})...\n")

!python train.py \
    --max-epochs 100 \
    --batch-size {RECOMMENDED_BATCH_SIZE} \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --patience 3

print("\n✓ Full training complete!")

## Option C: Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning with Optuna
N_TRIALS = 20  # Increase to 50 for production

print(f"Starting hyperparameter tuning ({N_TRIALS} trials)...")
print("⏱️  This will take 1-5 hours depending on GPU.\n")

!python tune_hyperparameters.py \
    --n-trials {N_TRIALS} \
    --output tuned_hyperparameters.json

# Show best hyperparameters
import json
with open('tuned_hyperparameters.json') as f:
    tuned = json.load(f)

print("\n" + "="*80)
print("BEST HYPERPARAMETERS")
print("="*80)
for key, value in tuned['hyperparameters'].items():
    print(f"  {key}: {value}")
print(f"\nBest val_loss: {tuned['val_loss']:.4f}")

In [ ]:
# Train with tuned hyperparameters
print("Training with tuned hyperparameters...\n")

!python train.py \
    --load-hyperparameters tuned_hyperparameters.json \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --max-epochs 100 \
    --patience 3

print("\n✓ Training with tuned hyperparameters complete!")

## Option D: Weighted Loss Verification

In [ ]:
# Verify weighted loss mechanism
!python verify_weighted_loss.py

# 4. View Results

In [ ]:
# View training metadata
import json
from pathlib import Path

metadata_path = Path(CHECKPOINT_DIR) / "training_metadata.json"

if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
    
    print("="*80)
    print("TRAINING METADATA")
    print("="*80)
    print(f"\nTraining Date: {metadata['training_date']}")
    
    print("\nHyperparameters:")
    for key, value in metadata['hyperparameters'].items():
        print(f"  {key}: {value}")
    
    print(f"\nDataset: {metadata['dataset_info']['total_samples']} samples")
    print(f"  Train: {metadata['dataset_info']['train_samples']}")
    print(f"  Val: {metadata['dataset_info']['val_samples']}")
    
    print(f"\nFeatures ({len(metadata['features'])}):")
    for feat in metadata['features']:
        print(f"  - {feat}")
else:
    print("⚠️  No training metadata found. Run training first.")

In [ ]:
# Download checkpoint (if not using Google Drive)
if not MOUNT_DRIVE:
    from google.colab import files
    
    print("Downloading checkpoints...\n")
    
    best_ckpt = Path(CHECKPOINT_DIR) / "best_model.ckpt"
    if best_ckpt.exists():
        files.download(str(best_ckpt))
        print("✓ Downloaded: best_model.ckpt")
    
    if metadata_path.exists():
        files.download(str(metadata_path))
        print("✓ Downloaded: training_metadata.json")
else:
    print(f"✓ Checkpoints saved to Google Drive: {CHECKPOINT_DIR}")
    print("  Access them anytime from your Drive!")

# Training Time Estimates\n\n**T4 GPU (Colab Free):**\n- Quick test: 5-10 min\n- Full training: 40-80 min\n- Tuning (20 trials): 4-8 hours\n\n**A100 GPU (Colab Pro+):**\n- Quick test: 2-3 min\n- Full training: 10-20 min\n- Tuning (20 trials): 1-2 hours\n\n**Tips:**\n- Start with Option A (quick test) first\n- Use Google Drive to preserve checkpoints\n- Monitor GPU: `!nvidia-smi`